# 01 — EDGAR fetch

Search EDGAR for 10-K filings and download their primary documents to `raw/`.

**Idempotent:** a filing already in `raw/` is skipped. Safe to re-run after an interruption.

Read the `edgar-harvesting` skill before changing anything here.

In [ ]:
# Colab bootstrap. Run once per runtime.
!pip install -q google-cloud-storage beautifulsoup4 jsonschema

from google.colab import auth
auth.authenticate_user()

import sys, pathlib
GITHUB_USER = ''   # TODO
REPO = pathlib.Path('/content/auditagent-bench')
if not REPO.exists():
    !git clone -q https://github.com/{GITHUB_USER}/auditagent-bench.git {REPO}
sys.path.insert(0, str(REPO))

In [ ]:
# --- Config. Every tunable value in this notebook lives in this cell. ---
BUCKET = ''            # TODO: GCS bucket name
RAW_PREFIX = 'raw'

# Sample definition. Recorded in the run manifest so the sample is reproducible.
QUERY = '"material weakness"'
FORMS = '10-K'
DATE_FROM = '2025-01-01'
DATE_TO = '2026-12-31'
TARGET_N = 250

In [ ]:
from src import edgar, gcs
from datetime import datetime, timezone
import json

manifest = {
    'stage': '01_edgar_fetch',
    'query': QUERY,
    'forms': FORMS,
    'date_from': DATE_FROM,
    'date_to': DATE_TO,
    'started_at': datetime.now(timezone.utc).isoformat(),
}
print(json.dumps(manifest, indent=2))

In [ ]:
# Collect candidate filings from full-text search.
candidates = {}
for hit in edgar.full_text_search(QUERY, forms=FORMS, date_from=DATE_FROM, date_to=DATE_TO):
    src = hit['_source']
    accession = edgar.normalize_accession(hit['_id'].split(':')[0])
    candidates[accession] = {
        'accession_number': accession,
        'cik': str(src['ciks'][0]).lstrip('0'),
        'company_name': src['display_names'][0],
        'filing_date': src['file_date'],
        'form_type': src['root_form'],
    }
    if len(candidates) >= TARGET_N:
        break

print(f'{len(candidates)} candidate filings')

In [ ]:
# Fetch. Skip anything already in raw/ — this is what makes the stage resumable.
fetched = skipped = failed = 0
failures = []

for accession, meta in candidates.items():
    path = f'{RAW_PREFIX}/{accession}.html'
    if gcs.blob_exists(BUCKET, path):
        skipped += 1
        continue
    try:
        url = edgar.primary_document(meta['cik'], accession)
        gcs.write_text(BUCKET, path, edgar.get(url).text, content_type='text/html')
        gcs.write_json(BUCKET, f'{RAW_PREFIX}/meta/{accession}.json', {**meta, 'source_url': url})
        fetched += 1
    except Exception as exc:
        failed += 1
        failures.append({'accession_number': accession, 'error': repr(exc)})

print(f'fetched={fetched} skipped={skipped} failed={failed}')
if failures:
    gcs.write_json(BUCKET, f'{RAW_PREFIX}/_fetch_failures.json', failures)
    print(f'{len(failures)} failures written to {RAW_PREFIX}/_fetch_failures.json')